# 23-20 · Тестируем перемещение и отмену

Практика к разделу [«Проверяем перемещение и отмену»](../../site/chapters/glava-23/23-25-testy-peremeshheniya.html). Использует настоящий пакет `safesort`.

## Reproducible local environment

```bash
git clone https://github.com/Cartesian-School/safesort.git
cd safesort
python3.14 -m venv .venv
source .venv/bin/activate
# Windows PowerShell: .venv\Scripts\Activate.ps1
python -m pip install -U pip
python -m pip install -e ".[dev]"
python -m pip install jupyter ipykernel
python -m ipykernel install --user --name safesort-py314 --display-name "SafeSort Python 3.14"
jupyter lab
```

Select the **SafeSort Python 3.14** kernel. The diagnostic cell below must
point into this `.venv` and the cloned `src/safesort` tree.

In [ ]:
import sys
import safesort

print(sys.executable)
print(safesort.__file__)

## Цель

Написать и запустить три теста в духе тех, что живут в `projects/python/safesort/tests/test_executor.py` и `test_manifest.py`: успешное перемещение, полная отмена и конфликт при восстановлении.

## Example — тесты apply_plan() и undo()

In [ ]:
import tempfile
from pathlib import Path

from safesort.config import Config
from safesort.scanner import scan
from safesort.planner import build_plan
from safesort.executor import apply_plan
from safesort.manifest import write_manifest, undo
from safesort.models import MoveOperation, SortPlan


def test_apply_plan_moves_file_to_destination(tmp_path):
    source = tmp_path / "otchet.pdf"
    source.write_text("...", encoding="utf-8")
    destination = tmp_path / "Sorted" / "documents" / "otchet.pdf"
    plan = SortPlan(root=tmp_path, operations=(MoveOperation(source, destination),))

    results = apply_plan(plan)

    assert results[0].completed is True
    assert not source.exists()
    assert destination.exists()


def test_undo_restores_original_location(tmp_path):
    source = tmp_path / "otchet.pdf"
    source.write_text("...", encoding="utf-8")
    nastrojki = Config()
    plan = build_plan(scan(tmp_path, nastrojki), tmp_path, nastrojki)
    moves = apply_plan(plan)
    manifest_obj, _ = write_manifest(tmp_path, moves)

    result = undo(manifest_obj)

    assert source.exists()
    assert result.conflicts == ()


def test_undo_refuses_to_overwrite_conflict(tmp_path):
    source = tmp_path / "otchet.pdf"
    source.write_text("оригинал", encoding="utf-8")
    nastrojki = Config()
    plan = build_plan(scan(tmp_path, nastrojki), tmp_path, nastrojki)
    moves = apply_plan(plan)
    manifest_obj, _ = write_manifest(tmp_path, moves)

    source.write_text("кто-то создал новый файл здесь", encoding="utf-8")
    result = undo(manifest_obj)

    assert len(result.conflicts) == 1
    assert source.read_text(encoding="utf-8") == "кто-то создал новый файл здесь"


for test_func in (
    test_apply_plan_moves_file_to_destination,
    test_undo_restores_original_location,
    test_undo_refuses_to_overwrite_conflict,
):
    with tempfile.TemporaryDirectory() as tmp:
        test_func(Path(tmp))
    print(f"OK: {test_func.__name__}")

## Starter

Заполните отмеченное место. Неизменённый starter не проходит tests.

In [ ]:
def test_apply_plan_reports_missing_source(tmp_path):
    # TODO: build a plan for a path that was never created and assert failure.
    raise NotImplementedError


## Task

Допишите тест отсутствующего source: apply_plan должен вернуть completed=False и текст ошибки.

## Tests

Запустите после task cell: есть основной пример и хотя бы один крайний случай.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    test_apply_plan_reports_missing_source(Path(tmp))

# Edge case: an empty plan is a no-op.
with tempfile.TemporaryDirectory() as tmp:
    empty_root = Path(tmp)
    assert apply_plan(SortPlan(root=empty_root, operations=())) == []
    assert list(empty_root.iterdir()) == []
print("Tests passed")

## Hint

Создайте только объекты Path и SortPlan; сам source на диске создавать не нужно.

## Solution

<details><summary>Показать решение после собственной попытки</summary>

```python
def test_apply_plan_reports_missing_source(tmp_path):
    source = tmp_path / "prizrak.pdf"  # файла никогда не было
    destination = tmp_path / "Sorted" / "documents" / "prizrak.pdf"
    plan = SortPlan(root=tmp_path, operations=(MoveOperation(source, destination),))

    results = apply_plan(plan)

    assert results[0].completed is False
    assert results[0].error is not None


with tempfile.TemporaryDirectory() as tmp:
    test_apply_plan_reports_missing_source(Path(tmp))
print("OK: test_apply_plan_reports_missing_source")
```

</details>